In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [35]:
from pathlib import Path

PROJECT_ROOT_STR = "/content/drive/MyDrive/Indic-Multimodal-NMT"

PROJECT_ROOT = Path(
    PROJECT_ROOT_STR
)

print(PROJECT_ROOT)

print(f"Does '{PROJECT_ROOT}' exist? {PROJECT_ROOT.exists()}")

if PROJECT_ROOT.exists():
    print(f"Contents of '{PROJECT_ROOT}':")
    for item in PROJECT_ROOT.iterdir():
        print(item.name)
else:
    print(f"The folder '{PROJECT_ROOT}' does not exist. Please ensure Google Drive is mounted correctly and the path is accurate.")

/content/drive/MyDrive/Indic-Multimodal-NMT
Does '/content/drive/MyDrive/Indic-Multimodal-NMT' exist? True
Contents of '/content/drive/MyDrive/Indic-Multimodal-NMT':
data
Dataset Exploration.ipynb
Tokenizer.ipynb


In [11]:
import os

print(os.getcwd())

/content


In [14]:
import pandas as pd

df = pd.read_csv(
    PROJECT_ROOT / "data/processed/flickr30k_metadata.csv"
)

print(df.shape)

(155070, 3)


In [15]:
df.head()

,image_id,filename,source_text
0,0,1000092795.jpg,Two young guys with shaggy hair look at their ...
1,0,1000092795.jpg,"Two young, White males are outside near many b..."
2,0,1000092795.jpg,Two men in green shirts are standing in a yard.
3,0,1000092795.jpg,A man in a blue shirt standing in a garden.
4,0,1000092795.jpg,Two friends enjoy time spent together.


In [18]:
from pathlib import Path

TOKENIZER_DIR = PROJECT_ROOT / Path("data/tokenizers")
TOKENIZER_DIR.mkdir(parents=True, exist_ok=True)

corpus_path = TOKENIZER_DIR / "corpus.txt"

with open(corpus_path, "w", encoding="utf-8") as f:
    for sentence in df["source_text"]:
        f.write(sentence.strip() + "\n")

print(f"Corpus saved to: {corpus_path}")
print(f"Total sentences: {len(df)}")

Corpus saved to: /content/drive/MyDrive/Indic-Multimodal-NMT/data/tokenizers/corpus.txt
Total sentences: 155070


In [19]:
with open(corpus_path, "r", encoding="utf-8") as f:
    for _ in range(5):
        print(next(f).strip())

Two young guys with shaggy hair look at their hands while hanging out in the yard.
Two young, White males are outside near many bushes.
Two men in green shirts are standing in a yard.
A man in a blue shirt standing in a garden.
Two friends enjoy time spent together.


In [ ]:
VOCAB_SIZE = 16000

In [24]:
import sentencepiece as spm

SP_MODEL_PATH = TOKENIZER_DIR / Path("sentencepiece_16k")

spm.SentencePieceTrainer.train(
    input=str(corpus_path),
    model_prefix=SP_MODEL_PATH,
    vocab_size=16000,
    character_coverage=1.0,
    model_type="unigram"
)

print("SentencePiece training completed!")

SentencePiece training completed!


In [27]:
sp = spm.SentencePieceProcessor()

sp.load("/content/drive/MyDrive/Indic-Multimodal-NMT/data/tokenizers/sentencepiece_16k.model"
)

print("Vocabulary size:", sp.get_piece_size())

Vocabulary size: 16000


In [28]:
samples = [
    "A young boy is playing with a dog",
    "Two men are standing in a garden",
    "The child is wearing a red shirt"
]


for text in samples:
    tokens = sp.encode(
        text,
        out_type=str
    )

    print("\nText:")
    print(text)

    print("Tokens:")
    print(tokens)


Text:
A young boy is playing with a dog
Tokens:
['▁A', '▁young', '▁boy', '▁is', '▁playing', '▁with', '▁a', '▁dog']

Text:
Two men are standing in a garden
Tokens:
['▁Two', '▁men', '▁are', '▁standing', '▁in', '▁a', '▁garden']

Text:
The child is wearing a red shirt
Tokens:
['▁The', '▁child', '▁is', '▁wearing', '▁a', '▁red', '▁shirt']


In [29]:
def sentencepiece_stats(text):

    tokens = sp.encode(
        text,
        out_type=str
    )

    words = text.split()

    return len(words), len(tokens)


df["sp_stats"] = df["source_text"].apply(sentencepiece_stats)

df["word_count"] = df["sp_stats"].apply(lambda x: x[0])
df["token_count"] = df["sp_stats"].apply(lambda x: x[1])


df.head()

,image_id,filename,source_text,sp_stats,word_count,token_count
0,0,1000092795.jpg,Two young guys with shaggy hair look at their ...,"(16, 17)",16,17
1,0,1000092795.jpg,"Two young, White males are outside near many b...","(9, 11)",9,11
2,0,1000092795.jpg,Two men in green shirts are standing in a yard.,"(10, 11)",10,11
3,0,1000092795.jpg,A man in a blue shirt standing in a garden.,"(10, 11)",10,11
4,0,1000092795.jpg,Two friends enjoy time spent together.,"(6, 7)",6,7


In [31]:
print(
    "Average words:",
    df["word_count"].mean()
)

print(
    "Average SentencePiece tokens:",
    df["token_count"].mean()
)

fragmentation_ratio = df["token_count"].mean() / df["word_count"].mean()
print(
    "Fragmentation ratio:",
    fragmentation_ratio
)
# Lower fragmentation generally means better segmentation.

if fragmentation_ratio <= 1.2:
    print(f"A fragmentation ratio of {fragmentation_ratio:.2f} is generally considered low, indicating good segmentation.")
elif fragmentation_ratio <= 1.5:
    print(f"A fragmentation ratio of {fragmentation_ratio:.2f} is moderate, which might be acceptable depending on the specific use case.")
else:
    print(f"A fragmentation ratio of {fragmentation_ratio:.2f} is relatively high, suggesting that the tokenizer might be breaking words into too many subword units.")

Average words: 12.271948152447282
Average SentencePiece tokens: 13.625530405623268
Fragmentation ratio: 1.1102988894966976
A fragmentation ratio of 1.11 is generally considered low, indicating good segmentation.


In [33]:

df[
    [
        "filename",
        "source_text",
        "word_count",
        "token_count"
    ]
].to_csv(
    PROJECT_ROOT / "data/processed/sentencepiece_statistics.csv",
    index=False
)

In [38]:
#bpe

from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.normalizers import NFKC

bpe_tokenizer = Tokenizer(BPE(unk_token="[UNK]")) # unknown_token="[UNK]"

bpe_tokenizer.normalizer = NFKC() # NFKC = Unicode Normalization Form Compatibility Composition.
# It cleans up text so visually similar characters become consistent.

bpe_tokenizer.pre_tokenizer = Whitespace()


trainer = BpeTrainer(
    vocab_size=16000,
    special_tokens=[
        "[UNK]",
        "[PAD]",
        "[BOS]",
        "[EOS]"
    ]
)


bpe_tokenizer.train(
    files=[str(corpus_path)],
    trainer=trainer
)


bpe_path = PROJECT_ROOT_STR + "/data/tokenizers/bpe_16k.json"

bpe_tokenizer.save(bpe_path)

print("BPE tokenizer saved")

BPE tokenizer saved


In [40]:
from tokenizers import Tokenizer

bpe = Tokenizer.from_file(
    bpe_path
)

In [41]:
samples = [
    "A young boy is playing with a dog",
    "Two men are standing in a garden"
]


for text in samples:

    output = bpe.encode(text)

    print("\nText:")
    print(text)

    print("Tokens:")
    print(output.tokens)


Text:
A young boy is playing with a dog
Tokens:
['A', 'young', 'boy', 'is', 'playing', 'with', 'a', 'dog']

Text:
Two men are standing in a garden
Tokens:
['Two', 'men', 'are', 'standing', 'in', 'a', 'garden']


In [44]:
def bpe_stats(text):

    tokens = bpe.encode(text).tokens

    words = text.split()

    return len(words), len(tokens)

df["bpe_stats"] = df["source_text"].apply(bpe_stats)

df["bpe_word_count"] = (
    df["bpe_stats"]
    .apply(lambda x:x[0])
)

df["bpe_token_count"] = (
    df["bpe_stats"]
    .apply(lambda x:x[1])
)

In [45]:
bpe_avg_tokens = df["bpe_token_count"].mean()

bpe_fragmentation = (
    df["bpe_token_count"].mean()
    /
    df["bpe_word_count"].mean()
)


print("Average BPE tokens:", bpe_avg_tokens)

print(
    "BPE fragmentation:",
    bpe_fragmentation
)

Average BPE tokens: 13.634062036499646
BPE fragmentation: 1.1109941035548403


In [47]:
# Train WordPiece Tokenizer

from tokenizers.models import WordPiece
from tokenizers.trainers import WordPieceTrainer

wp_tokenizer = Tokenizer(
    WordPiece(
        unk_token="[UNK]"
    )
)


wp_tokenizer.normalizer = NFKC()

wp_tokenizer.pre_tokenizer = Whitespace()


wp_trainer = WordPieceTrainer(
    vocab_size=16000,
    special_tokens=[
        "[UNK]",
        "[PAD]",
        "[BOS]",
        "[EOS]"
    ]
)

wp_tokenizer.train(
    files=[str(corpus_path)],
    trainer=wp_trainer
)


wp_path = PROJECT_ROOT_STR + "/data/tokenizers/wordpiece_16k.json"

wp_tokenizer.save(wp_path)

print("WordPiece tokenizer saved")

WordPiece tokenizer saved


In [50]:
wp = Tokenizer.from_file(
    wp_path
)

def wp_stats(text):

    tokens = wp.encode(text).tokens

    words = text.split()

    return len(words), len(tokens)

df["wp_stats"] = df["source_text"].apply(wp_stats)


df["wp_word_count"] = (
    df["wp_stats"].apply(lambda x:x[0])
)


df["wp_token_count"] = (
    df["wp_stats"].apply(lambda x:x[1])
)

In [51]:
wp_avg_tokens = df["wp_token_count"].mean()

wp_fragmentation = (
    df["wp_token_count"].mean()
    /
    df["wp_word_count"].mean()
)


print("Average WordPiece tokens:", wp_avg_tokens)

print(
    "WordPiece fragmentation:",
    wp_fragmentation
)

Average WordPiece tokens: 13.65841877861611
WordPiece fragmentation: 1.1129788529861362


In [53]:
results = pd.DataFrame(
    {
        "Tokenizer":[
            "SentencePiece",
            "BPE",
            "WordPiece"
        ],

        "Avg Tokens/Sentence":[
            df["token_count"].mean(),
            df["bpe_token_count"].mean(),
            df["wp_token_count"].mean()
        ],

        "Fragmentation Ratio":[
            df["token_count"].mean()/df["word_count"].mean(),
            df["bpe_token_count"].mean()/df["bpe_word_count"].mean(),
            df["wp_token_count"].mean()/df["wp_word_count"].mean()
        ]
    }
)


results
# selected SentencePiece as it has the lowest fragmentation ratio

,Tokenizer,Avg Tokens/Sentence,Fragmentation Ratio
0,SentencePiece,13.625530,1.110299
1,BPE,13.634062,1.110994
2,WordPiece,13.658419,1.112979
